## Colab 준비
Colab에서 열었다면 **아래 칸을 맨 먼저 실행**하세요. 도우미 파일과 한글 글꼴을 받아요 (10~20초).

- 더 빠르게: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
- 고친 내용 저장: **파일 → Drive에 사본 저장**
- 맥 실습실에서 열었다면 이 칸은 아무 일도 안 해요.

In [ ]:
import os, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/sangwon9401/silicon-descent/main/colab/"
    for f in ['common.py', 'solutions/__init__.py', 'solutions/sol00.py', 'solutions/sol01.py', 'solutions/sol02.py', 'solutions/sol04.py']:
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + f, f)
    os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    print("Colab 준비 완료 ✅")
else:
    print("Colab이 아니에요. 그냥 넘어가요.")

# 04. 배치 샘플러 만들기

**목표**: 요청마다 temperature, top-k, top-p가 **다른** 배치를 한 번에 처리하는 샘플러를 직접 만들어요. 실제 서빙 엔진이 하는 일과 같아요.

사이트의 강의 "샘플링 직접 구현하기"를 먼저 보고 오세요.

모든 함수의 입력 `logits`는 `[B, V]` 모양이에요. B는 배치(요청 수), V는 토큰 종류 수예요. 설정값들은 `[B]` 모양 벡터예요.

In [ ]:
import torch
from common import *
from solutions import sol04 as ans

device = pick_device()
torch.manual_seed(0)
B, V = 4, 10
logits = torch.randn(B, V, device=device) * 2
temperature = torch.tensor([0.5, 1.0, 1.5, 1.0], device=device)
top_k = torch.tensor([3, 10, 5, 1], device=device)
top_p = torch.tensor([1.0, 0.8, 0.5, 1.0], device=device)
print("logits 모양:", tuple(logits.shape))

## 1. Temperature

요청마다 자기 temperature로 나눠요. `logits`는 `[B, V]`, `temperature`는 `[B]`라서, 모양을 `[B, 1]`로 바꿔 나눠야 해요: `temperature[:, None]`

In [ ]:
def apply_temperature(logits, temperature):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol04 import apply_temperature

In [ ]:
run_check("temperature", lambda: check_close("temperature", apply_temperature(logits, temperature), ans.apply_temperature(logits, temperature)))

## 2. Top-k

요청마다 **k번째로 큰 값보다 작은 것**을 `-inf`로 바꿔요. (`-inf`는 softmax 후 확률 0이 돼요.)

**힌트**
1. 큰 순서로 정렬: `sorted_vals, _ = logits.sort(dim=-1, descending=True)`
2. 줄마다 k번째 값 꺼내기 (0부터 세니까 k-1번째): `kth = sorted_vals.gather(1, (top_k - 1)[:, None])` → `[B, 1]`
3. `logits.masked_fill(logits < kth, float("-inf"))`

In [ ]:
def apply_top_k(logits, top_k):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol04 import apply_top_k

In [ ]:
run_check("top-k", lambda: check_close("top-k", apply_top_k(logits, top_k), ans.apply_top_k(logits, top_k)))

## 3. Top-p

가장 어려운 부분이에요. 확률이 큰 순서로 더해서, **내 앞까지의 합이 이미 p 이상이면** 나를 버려요. (그래야 1등은 항상 남아요.)

**힌트**
1. 정렬하면서 원래 위치도 기억: `sorted_vals, idx = logits.sort(dim=-1, descending=True)`
2. 확률과 누적합: `probs = sorted_vals.softmax(-1)`, `cum = probs.cumsum(-1)`
3. 버릴 자리: `remove = (cum - probs) >= top_p[:, None]`
4. 정렬된 상태에서 버릴 자리를 `-inf`로
5. 원래 순서로 되돌리기: `out = torch.empty_like(logits); out.scatter_(1, idx, sorted_vals)`

In [ ]:
def apply_top_p(logits, top_p):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol04 import apply_top_p

In [ ]:
run_check("top-p", lambda: check_close("top-p", apply_top_p(logits, top_p), ans.apply_top_p(logits, top_p)))

## 4. 전부 합치기

순서: temperature → top-k → top-p → softmax → 확률대로 하나 뽑기 `torch.multinomial(probs, 1).squeeze(1)`

**특별 규칙**: temperature가 0 이하인 요청은 greedy(가장 큰 logit)로 처리해요. 0으로 나누면 안 되니까 그 줄은 temperature를 1로 바꿔 계산하고, 마지막에 `torch.where`로 greedy 결과를 골라요.

In [ ]:
def sample(logits, temperature, top_k, top_p):
    ...  # 여기를 채우세요


# 막히면:
# from solutions.sol04 import sample

In [ ]:
def check_sample():
    # (1) temperature 0인 줄은 항상 argmax여야 해요.
    t0 = torch.tensor([0.0, 0.0, 1.0, 1.0], device=device)
    out = sample(logits, t0, torch.full((B,), V, device=device), torch.ones(B, device=device))
    if not check_equal("greedy 줄", out[:2].tolist(), logits[:2].argmax(-1).tolist()):
        return False
    # (2) top_k = 1이면 무조건 1등이어야 해요.
    out = sample(logits, torch.ones(B, device=device), torch.ones(B, dtype=torch.long, device=device), torch.ones(B, device=device))
    if not check_equal("top-k = 1", out.tolist(), logits.argmax(-1).tolist()):
        return False
    # (3) 여러 번 뽑은 빈도가 이론 확률과 비슷해야 해요.
    row = logits[1:2].repeat(4000, 1)
    out = sample(row, torch.ones(4000, device=device), torch.full((4000,), V, device=device), torch.ones(4000, device=device))
    freq = torch.bincount(out, minlength=V).float() / 4000
    return check_close("뽑기 빈도 ≈ 확률", freq, logits[1].softmax(-1), atol=0.03)

run_check("sample", check_sample)

## 5. 진짜 모델로 비교해 보기

SmolLM2에 같은 프롬프트를 주고, 설정만 바꿔서 30토큰씩 생성해요. 내가 만든 `sample` 함수를 그대로 써요.

In [ ]:
model, tok = load_model(device)
prompt = "Once upon a time, in a small village,"
settings = {"greedy (T=0)": (0.0, 50, 1.0), "보통 (T=0.8, top-p 0.9)": (0.8, 50, 0.9), "창의적 (T=1.5)": (1.5, 0, 1.0)}

@torch.no_grad()
def generate(prompt, T, k, p, n=30):
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    kk = k if k > 0 else model.config.vocab_size
    for _ in range(n):
        lg = model(ids).logits[:, -1, :]
        nxt = sample(lg, torch.tensor([T], device=device), torch.tensor([kk], device=device), torch.tensor([p], device=device))
        ids = torch.cat([ids, nxt[:, None]], dim=1)
    return tok.decode(ids[0])

for name, (T, k, p) in settings.items():
    print(f"[{name}]\n{generate(prompt, T, k, p)}\n")

## 정리

- 서빙 엔진은 배치 안의 요청마다 설정이 달라도 **한 번의 텐서 연산**으로 처리해요. 방금 만든 게 바로 그 구조예요.
- top-p의 `sort`는 어휘가 10만 개가 넘으면 꽤 비싸요. 그래서 vLLM, SGLang은 정렬을 피하는 특수 커널(FlashInfer 샘플링 등)을 써요. 4단계에서 다시 만나요.

1단계 실습은 여기까지예요. 사이트로 돌아가서 진도를 체크하고 2단계로 넘어가세요!